# Multi-board replication (leave-one-board-out) on Colab A100

Use only the rendered handoff notebook in a fresh Colab **A100** runtime and Run all. It loops over
every fold marked `run: true` in `configs/lobo/folds.yaml`, running the unchanged paired pipeline
(data gate, GPU gates, six hash-locked runs, one-shot final evaluation, deployment gate, verifiable
package) once per held-out board. Every step skips completed work, so after a disconnect simply
Run all again. Do not edit any cell.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

os.environ['YOLO_AUTOINSTALL'] = 'false'
os.environ['ULTRALYTICS_SKIP_REQUIREMENTS_CHECKS'] = '1'
os.environ['MPLBACKEND'] = 'Agg'
SOURCE_BUNDLE_SHA256 = 'PASTE_FINAL_BUNDLE_SHA256'
EXPECTED_GIT_SHA = 'PASTE_FINAL_GIT_SHA'
LOBO_HANDOFF_DIRECTORY = 'PASTE_LOBO_HANDOFF_DIRECTORY'
immutable_values = (SOURCE_BUNDLE_SHA256, EXPECTED_GIT_SHA, LOBO_HANDOFF_DIRECTORY)
if any(value.startswith('PASTE' + '_') for value in immutable_values):
    raise RuntimeError('Use the rendered LOBO handoff notebook without manual editing')
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/pcb-defect-paired')
SOURCE_BUNDLE = Path(LOBO_HANDOFF_DIRECTORY) / 'pcb-defect-source.bundle'
REPO = Path('/content/pcb-defect-lobo-source')
DATASET = DRIVE_ROOT / 'dataset' / 'pcb'
WORKSPACE_ROOT = DRIVE_ROOT / 'workspaces'
PACKAGE_ROOT = DRIVE_ROOT / 'packages'
print({'bundle': str(SOURCE_BUNDLE), 'dataset': str(DATASET), 'workspaces': str(WORKSPACE_ROOT)})

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if sha256_file(SOURCE_BUNDLE) != SOURCE_BUNDLE_SHA256:
    raise RuntimeError('Source bundle SHA-256 mismatch')
if REPO.exists():
    observed = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO, check=True, capture_output=True, text=True).stdout.strip()
    if observed != EXPECTED_GIT_SHA:
        raise RuntimeError('Existing /content checkout has the wrong Git SHA; restart runtime')
else:
    subprocess.run(['git', 'clone', str(SOURCE_BUNDLE), str(REPO)], check=True)
    subprocess.run(['git', 'checkout', '--detach', EXPECTED_GIT_SHA], cwd=REPO, check=True)
status = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO, check=True, capture_output=True, text=True).stdout
if status:
    raise RuntimeError(f'Source checkout is dirty: {status}')
print('SOURCE GATE PASS', EXPECTED_GIT_SHA)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.11.18'], check=True)
UV = shutil.which('uv')
if not UV:
    raise RuntimeError('uv installation failed')
subprocess.run([UV, 'sync', '--locked', '--no-editable', '--reinstall-package', 'pcb-defect', '--extra', 'train', '--group', 'eval'], cwd=REPO, check=True)
VENV_PYTHON = REPO / '.venv' / 'bin' / 'python'
if not VENV_PYTHON.is_file():
    raise RuntimeError(f'Locked environment Python is missing: {VENV_PYTHON}')
def runtime_contract_state(label: str) -> dict[str, object]:
    command = [
        str(VENV_PYTHON),
        '-m',
        'pcb_defect.runtime_contract',
        '--require-cuda-provider',
    ]
    result = subprocess.run(command, cwd=REPO, text=True, capture_output=True)
    print(f'[{label}] returncode={result.returncode}')
    if result.stdout:
        print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='', file=sys.stderr)
    if result.returncode:
        raise RuntimeError(f'{label} FAILED')
    lines = [line for line in result.stdout.splitlines() if line.strip()]
    if not lines:
        raise RuntimeError(f'{label} returned no runtime state')
    try:
        return json.loads(lines[-1])
    except json.JSONDecodeError as exc:
        raise RuntimeError(f'{label} returned invalid runtime JSON') from exc

LOCKED_RUNTIME_STATE = runtime_contract_state('LOCKED RUNTIME CONTRACT')
sys.path.insert(0, str(REPO / 'src'))
from pcb_defect.notebook_runtime import run_streaming_command
def import_probe(label: str, code: str) -> None:
    result = subprocess.run([str(VENV_PYTHON), '-c', code], cwd=REPO, text=True, capture_output=True)
    print(f'[{label}] returncode={result.returncode}')
    if result.stdout:
        print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='', file=sys.stderr)
    if result.returncode:
        raise RuntimeError(f'IMPORT PROBE FAILED: {label}')
import_probe('python', "import sys, numpy; print(sys.version); print('numpy', numpy.__version__)")
import_probe('torch-cuda', "import torch; print('torch', torch.__version__); print('cuda', torch.version.cuda); print('available', torch.cuda.is_available()); assert torch.cuda.is_available(); name=torch.cuda.get_device_name(0); print('gpu', name); assert 'A100' in name")
import_probe('ultralytics', "import ultralytics; print('ultralytics', ultralytics.__version__)")
def run_logged(label, command, log_path):
    result = subprocess.run(command, cwd=REPO, text=True, capture_output=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_path.write_text(result.stdout + '\n--- STDERR ---\n' + result.stderr, encoding='utf-8')
    print(f'[{label}] returncode={result.returncode}; log={log_path}')
    if result.stdout:
        print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='', file=sys.stderr)
    return result
print('LOCKED ENVIRONMENT GATE PASS')

In [ ]:
if DATASET.exists() and not (DATASET / 'conversion_report.json').is_file():
    raise RuntimeError('Partial dataset directory exists; inspect it instead of overwriting')
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([str(VENV_PYTHON), '-m', 'pcb_defect.data_prep.prepare', '--out', str(DATASET), '--strategy', 'grouped', '--seed', '42'], cwd=REPO, check=True)
registry_check = subprocess.run([str(VENV_PYTHON), '-m', 'pcb_defect.lobo', 'folds', '--repo', str(REPO), '--dataset', str(DATASET)], cwd=REPO, text=True, capture_output=True)
print(registry_check.stdout, end='')
if registry_check.returncode:
    print(registry_check.stderr, end='', file=sys.stderr)
    raise RuntimeError('FOLD REGISTRY VERIFICATION FAILED')
import yaml
REGISTRY = yaml.safe_load((REPO / 'configs' / 'lobo' / 'folds.yaml').read_text(encoding='utf-8'))
FOLDS = [fold for fold in REGISTRY['folds'] if fold['run']]
print('PENDING FOLDS', [fold['board'] for fold in FOLDS])
print('DATA AND FOLD REGISTRY GATE PASS')

In [ ]:
def package_is_verified(package: Path) -> bool:
    sidecar = package.with_suffix(package.suffix + '.sha256')
    if package.exists() != sidecar.exists():
        raise RuntimeError('Partial result package exists; inspect it instead of overwriting')
    if not package.exists():
        return False
    fields = sidecar.read_text(encoding='ascii').split()
    if len(fields) != 2 or fields[1] != package.name or fields[0] != sha256_file(package):
        raise RuntimeError('Existing result package fails its SHA-256 sidecar')
    return True

def require_clean_checkout(stage: str) -> None:
    status = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO, check=True, capture_output=True, text=True).stdout
    if status:
        raise RuntimeError(f'Source checkout became dirty during {stage}: {status}')

completed = []
for fold in FOLDS:
    board = fold['board']
    workspace = WORKSPACE_ROOT / f'{EXPECTED_GIT_SHA[:12]}-board{board}'
    base_model = workspace / 'inputs' / 'base_model.pt'
    fold_config = REPO / fold['config']
    fold_artifacts = REPO / fold['artifacts']
    package = PACKAGE_ROOT / f'paired-results-a100-{EXPECTED_GIT_SHA[:12]}-board{board}.zip'
    print(f'===== FOLD board {board}: workspace={workspace} =====')
    if package_is_verified(package):
        print(f'SKIP completed fold board {board}: {package}')
        completed.append((board, package))
        continue
    common = ['--repo', str(REPO), '--dataset', str(DATASET), '--workspace', str(workspace)]
    protocol = ['--protocol-config', str(fold_config), '--protocol-artifacts', str(fold_artifacts)]
    subprocess.run([str(VENV_PYTHON), '-m', 'pcb_defect.data_prep.paired', '--source', str(DATASET), '--config', str(fold_config), '--artifacts', str(fold_artifacts), '--runtime', str(workspace / 'runtime_data')], cwd=REPO, check=True)
    require_clean_checkout(f'fold {board} data preparation')
    subprocess.run([str(VENV_PYTHON), '-m', 'pcb_defect.experiment', 'resolve-base', '--repo', str(REPO), '--workspace', str(workspace)], cwd=REPO, check=True)
    subprocess.run([str(VENV_PYTHON), '-m', 'pcb_defect.experiment', 'preflight', *common, *protocol, '--base-model', str(base_model), '--required-gpu', 'A100'], cwd=REPO, check=True)
    gate_log = workspace / 'gates_command.log'
    gate_result = run_logged(f'gates board {board}', [str(VENV_PYTHON), '-m', 'pcb_defect.experiment', 'gates', *common, *protocol, '--base-model', str(base_model), '--required-gpu', 'A100'], gate_log)
    if gate_result.returncode:
        gate_report = workspace / 'gates' / 'gate_report.json'
        if gate_report.is_file():
            print('PARTIAL GATE REPORT:', gate_report.read_text(encoding='utf-8'), file=sys.stderr)
        raise RuntimeError(f'GPU GATE COMMAND FAILED for board {board}; full log: {gate_log}')
    train_log = workspace / 'train_all_command.log'
    run_streaming_command([str(VENV_PYTHON), '-m', 'pcb_defect.experiment', 'train-all', *common, *protocol, '--base-model', str(base_model)], cwd=REPO, log_path=train_log, label=f'train-all board {board}')
    evaluation_log = workspace / 'final_evaluation_command.log'
    evaluation_result = run_logged(f'final evaluation board {board}', [str(VENV_PYTHON), '-m', 'pcb_defect.final_evaluation', *common, '--protocol-config', str(fold_config)], evaluation_log)
    if evaluation_result.returncode:
        raise RuntimeError(f'FINAL EVALUATION COMMAND FAILED for board {board}; full log: {evaluation_log}')
    deployment_log = workspace / 'deployment_command.log'
    deployment_runtime_before = runtime_contract_state(f'DEPLOYMENT RUNTIME BEFORE board {board}')
    deployment_result = run_logged(f'deployment board {board}', [str(VENV_PYTHON), '-m', 'pcb_defect.deployment', *common, '--protocol-config', str(fold_config)], deployment_log)
    deployment_runtime_after = runtime_contract_state(f'DEPLOYMENT RUNTIME AFTER board {board}')
    if deployment_runtime_after != deployment_runtime_before:
        raise RuntimeError('ONNX Runtime state changed across the deployment command')
    if deployment_result.returncode:
        for evidence in (workspace / 'deployment' / 'deployment_gate.json', workspace / 'deployment' / 'model_contract.candidate.json'):
            if evidence.is_file():
                print(f'DEPLOYMENT EVIDENCE {evidence.name}:', evidence.read_text(encoding='utf-8'), file=sys.stderr)
        raise RuntimeError(f'DEPLOYMENT COMMAND FAILED for board {board}; full log: {deployment_log}')
    package_log = workspace / 'result_package_command.log'
    package_result = run_logged(f'result package board {board}', [str(VENV_PYTHON), '-m', 'pcb_defect.result_package', '--workspace', str(workspace), '--output', str(package)], package_log)
    if package_result.returncode:
        raise RuntimeError(f'RESULT PACKAGE COMMAND FAILED for board {board}; full log: {package_log}')
    if not package_is_verified(package):
        raise RuntimeError(f'Result package for board {board} was not created atomically')
    completed.append((board, package))
    print(f'FOLD COMPLETE board {board}', package, sha256_file(package))
for board, package in completed:
    print('LOBO PACKAGE', board, package, sha256_file(package))
print('LOBO HANDOFF COMPLETE', len(completed), 'folds')